# Example notebook for training, plotting and generation

In [1]:
import numpy as np
import torch
import os
import sys

## Training

## Generation

### Load Q,K,V, calculate J

In [22]:
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, parent_dir)
from model import AttentionModel
from model_PCA_correlation import AttentionModel_PCA
from model_PCA_cond import ModelPCAcondJ
from dcascore import *
from utils import read_fasta_alignment, remove_duplicate_sequences, add_PCA_coords, quickread

# back to original path (in PLM)
sys.path.pop(0)  # Removes the parent_dir from sys.path
from model import AttentionModel

from plm_gen_methods import generate_plm_n_save, generate_coords_n_save, generate_multiple_targets_n_save,generate_plm_vect_n_save
from seq_utils import read_tensor_from_txt, set_seed, letters_to_nums, modify_seq 

In [4]:
"""
    Load Q, K, V matrices from jdoms (after training)
"""
set_seed()
H = 64
d= 10
N = 174
n_epochs = 250
nb_PCA_comp=2
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
cwd = parent_dir
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA35_cond/Q_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA35_cond/K_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA35_cond/V_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
H,d,N=Q_1.shape
q=V_1.shape[1]


In [5]:
model=ModelPCAcondJ(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)

i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
# Mask only part without CPA
H, _, M = K_1.shape #(M=N+m)

W[:,:,:-2] = W[:,:,:-2] * mask
    
# Compute Jtens
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
q = Jtens.shape[0]
N = Jtens.shape[2]
print(q)
print(N)
print(Jtens.shape)

torch.Size([64, 63, 65])
21
63
torch.Size([21, 56, 63, 65])


### Generate

In [18]:
class SequencePLMvec:
    def __init__(self, J, initial_sequence = None, beta = 1, nb_PCA_comp=0, target_coords=np.array([]), beta_PCA=1):
        """
        Initialize the SequencePLM object with a coupling tensor J of the family and an optional initial sequence.
        """
        self.J = J
        #self.L = J.shape[-1]
        self.beta = beta
        self.L = J.shape[-1]-nb_PCA_comp # Length of amino acid sequence (without PCA coords)
        if initial_sequence is None:
            self.sequence = np.random.choice(np.arange(21), self.L) # Sequence of ints (1 to 21) 
        else:
            self.sequence = initial_sequence
        if nb_PCA_comp > 0:
            if len(target_coords) != nb_PCA_comp:
                raise ValueError("Mismatch between nb_PCA_comp and length of target_coords")
            self.sequence = np.concatenate([self.sequence, target_coords])
        self.nb_PCA_comp = nb_PCA_comp

    def plm_calc(self, site, trial_aa):
        """
        Compute unnormalized pseudo-likelihood of trial_aa at a given site.
        site: int from 0 to L-1
        trial_aa: int from 0 to 21 (amino acid index)
        """
        if site < 0 or site >= self.L:
            raise ValueError(f"Site {site} is out of bounds for sequence length {self.L}.")
        sum_energy = 0.0
        for j in range(self.L):
            if j == site:
                continue
            aa_j = self.sequence[j]
            sum_energy += self.beta * self.J[trial_aa, aa_j, site, j] # check indexing
            #sum_energy += self.J[aa_j, trial_aa, j, site]
        prob = sum_energy
        return prob

    def plm_site_distribution(self, site):
        """
        Compute probability distriution for specific site (normalized)
        """
        probs = []
        for trial_aa in range(21):
            probs.append(self.plm_calc(site, trial_aa))
        probs = np.array(probs)
        probs = np.exp(probs-probs.max()) #to avoid overflow for high beta
        probs /= probs.sum()
        return probs
    
    def plm_site_distribution_vectorized_prev(self, site):
        """
        Compute normalized probability distribution for a specific site.
        Fully vectorized over amino acids.
        """
        if site < 0 or site >= self.L:
            raise ValueError(f"Site {site} is out of bounds for sequence length {self.L}.")

        # Indices of other sites
        mask = np.arange(self.L) != site
        other_sites = np.arange(self.L)[mask]
        aa_vector = self.sequence[mask]

        # Energies for all trial amino acids at once
        trial_aas = np.arange(21)[:, None]  # shape (21,1)
        energies = self.J[trial_aas, aa_vector, site, other_sites]  # shape (21, L-1)
        sum_energy = self.beta * np.sum(energies, axis=1)  # shape (21,)
        # Softmax with numerical stability
        # add contribution of PCA coords if nb_PCA_comp>0 (J of shape ([q, q+Nbins, L, L+nb_PCA_comp]))
        if self.nb_PCA_comp > 0:
            pca_indices = np.arange(self.L, self.L + self.nb_PCA_comp)
            pca_aas = self.sequence[pca_indices]
            pca_energies = self.J[trial_aas, pca_aas, site, pca_indices]  # shape (21, nb_PCA_comp)
            sum_energy += self.beta * np.sum(pca_energies, axis=1)  # shape (21,)   

        probs = np.exp(sum_energy - np.max(sum_energy))
        probs /= probs.sum()
        return probs
    
    def plm_site_distribution_vectorized(self, site):
        """
        Compute normalized probability distribution for a specific site.
        Combines sequence couplings and PCA couplings.
        Fully vectorized over amino acids.
        """
        if site < 0 or site >= self.L:
            raise ValueError(f"Site {site} is out of bounds for sequence length {self.L}.")
    
        q = 21  # number of amino acid states
    
        # -------------------
        # 1. Sequence couplings
        # -------------------
        mask = np.arange(self.L) != site
        other_sites = np.arange(self.L)[mask]
        aa_vector = self.sequence[:self.L][mask]   # only real sites
    
        trial_aas = np.arange(q)[:, None]             # (21,1) trial states for this site
        seq_energies = self.J[trial_aas, aa_vector, site, other_sites]  # (21, L-1)
        sum_energy = self.beta * np.sum(seq_energies, axis=1)           # (21,)
    
        # -------------------
        # 2. PCA couplings
        # -------------------
        if self.nb_PCA_comp > 0:
            pca_indices = np.arange(self.L, self.L + self.nb_PCA_comp)   # j-axis indices
            pca_bins = self.sequence[pca_indices]                        # which bin each PCA comp is in
            pca_energies = self.J[trial_aas, q + pca_bins, site, pca_indices]  # (21, nb_PCA_comp)
            sum_energy += self.beta * np.sum(pca_energies, axis=1)       # add to total
    
        # -------------------
        # 3. Softmax normalization
        # -------------------
        probs = np.exp(sum_energy - np.max(sum_energy))
        probs /= probs.sum()
    
        return probs
    
    def check_vectorized(self, site):
        """
        Check that vectorized and loop implementations give same result
        """
        probs_loop = self.plm_site_distribution(site)
        probs_vec = self.plm_site_distribution_vectorized(site)
        assert np.allclose(probs_loop, probs_vec), f"Mismatch at site {site}"
        print(f"Vectorized check passed at site {site}")
    
    def draw_aa(self, site, vec=True):
        """
        Sample a new AA at the given site from PLM distribution
        """
        if vec:
            probs = self.plm_site_distribution_vectorized(site)
        else:
            probs = self.plm_site_distribution(site)
        new_aa = np.random.choice(21, p=probs) # aa from 0 to 20
        self.sequence[site] = new_aa

    def update_PCA_coords(self, model, plot=False):
        """
        Jointly update both PCA coordinates using Boltzmann sampling
        based on PLM-derived J_PCA tensor and the current amino acid sequence.

        Assumes: nb_PCA_comp == 2
        """
        #if self.J_PCA is None:
        #    raise ValueError("J_PCA not provided in model.")
        #if self.nb_PCA_comp != 2:
        #    raise ValueError(f"Joint 2D PCA update only supports 2 PCA components, got {self.nb_PCA_comp}.")

        L = self.L
        Nbins = 35
        offset = L  # Start index of PCA coords in self.sequence
        probs_2D = np.zeros((Nbins, Nbins))
        energies_2D = self.compute_coord_energy(model)

        # Numerically stable softmax
        shifted_energies = self.beta_PCA * energies_2D #(energy should be -)
        #shifted_energies -= shifted_energies.max()

        shifted_energies -= shifted_energies.max()  # subtract max for numerical stability

        probs_2D = np.exp(-shifted_energies)
        total = probs_2D.sum()

        # Check for numerical issues
        if total == 0 or np.isnan(total) or np.isinf(total):
            probs_2D = np.ones_like(probs_2D) / probs_2D.size  # uniform distribution
        else:
            probs_2D /= total

        # Sample from joint distribution
        flat_probs = probs_2D.flatten()
        choice = np.random.choice(Nbins * Nbins, p=flat_probs)
        i_sampled, j_sampled = np.unravel_index(choice, (Nbins, Nbins))

        self.sequence[offset + 0] = i_sampled  # PCA comp 0
        self.sequence[offset + 1] = j_sampled  # PCA comp 1

        return np.array([i_sampled, j_sampled])

    def seq_energy(self):
        sum=0
        for i in range(self.L):
            for j in range(self.L):
                sum+=self.J[self.sequence[i], self.sequence[j],i,j]
        return sum

In [19]:
from tqdm import tqdm

def generate_plm(J, N_seqs=40000, init_sequence=None, beta=1, nb_PCA_comp=0, beta_pca = 0, target_coords=np.array([]), site_selector=None):
    """
    Generate N_seqs new sequences using PLM.

    Parameters
    ----------
    J : interaction tensor
    N_seqs : int
        Number of sequences to generate
    init_sequence : array-like
        Optional initial sequence
    beta : float
        Inverse temperature
    site_selector : function(seq) -> int
        Function to select site to update.
        If None, sites are chosen uniformly at random.
    """
    gen_sequences = []
    seq = SequencePLMvec(J, init_sequence, beta=beta, nb_PCA_comp=nb_PCA_comp, target_coords=target_coords, beta_PCA=beta_pca)

    for _ in tqdm(range(N_seqs)):
        if site_selector is None:
            site = np.random.randint(seq.L)  # uniform random
        else:
            site = site_selector(seq)        # custom site selection
        seq.draw_aa(site, vec=True)
        gen_sequences.append(seq.sequence.copy())

    return np.array(gen_sequences)

In [ ]:
class SequencePLMvec:
    def __init__(self, J, initial_sequence = None, beta = 1, nb_PCA_comp=0,PCA_component_list=np.array([]),J_tens_PCA=None,beta_PCA=1):
        """
        Initialize the SequencePLM object with a coupling tensor J of the family and an optional initial sequence.
        """
        self.J = J
        self.J_PCA=J_tens_PCA
        #self.L = J.shape[-1]
        self.beta = beta
        self.beta_PCA=beta_PCA
        self.nb_PCA_comp=nb_PCA_comp
        if not (J_tens_PCA is None):
            if nb_PCA_comp!=J_tens_PCA.shape[-1]:
                print("Mismatch of PCA tensor and nb PCA components indicated")
        if J_tens_PCA is None:
            self.L = J.shape[-1] - nb_PCA_comp  # Length of the sequence without PCA components
        else:
            self.L = J.shape[-1]
        if initial_sequence is None:
            self.sequence = np.random.choice(np.arange(21), self.L) # Sequence of ints (1 to 21) 
            if len(PCA_component_list)==nb_PCA_comp:
                self.sequence = np.concatenate((self.sequence,PCA_component_list))
            else:
                print("number of PCA components doesn't match size of PCA list")
        else:
            self.sequence = initial_sequence

    def to_letter(self):
        """
        Show sequence as letters
        """
        print("Sequence:", self.sequence)
        num_to_letter = {v: k for k, v in letter_to_num.items()}
        letter_seq = ''.join([num_to_letter[i] for i in self.sequence[:len(self.sequence)-self.nb_PCA_comp]])
        print(letter_seq)
        return letter_seq

    def plm_calc_vec(self, site, trial_aa):
        """
        Compute unnormalized pseudo-likelihood of trial_aa at a given site.
        site: int from 0 to L-1
        trial_aa: int from 0 to 21 (amino acid index)
        """
        if site < 0 or site >= self.L:
            raise ValueError(f"Site {site} is out of bounds for sequence length {self.L}.")
        sum_energy = 0.0
        if not ( self.J_PCA is None):
            #for i in range(self.nb_PCA_comp):
            #    sum_energy+= self.J_PCA[trial_aa,self.sequence[i],site,i]
            # PREBIOUS
            #for i in range(self.nb_PCA_comp):
            #    PCA_coord = self.sequence[self.L + i]  # the PCA coordinate at component i
            #    sum_energy += self.beta_PCA*self.J_PCA[trial_aa, PCA_coord, site, i]
            # NEW
            if self.nb_PCA_comp == 1:       
                joint_bin = self.sequence[self.L]  
                sum_energy += self.beta_PCA * self.J_PCA[trial_aa, joint_bin, site, 0]
            else:                           
                for i in range(self.nb_PCA_comp):
                    PCA_coord = self.sequence[self.L + i] 
                    sum_energy += self.beta_PCA * self.J_PCA[trial_aa, PCA_coord, site, i]
        else:
            for i in range(self.nb_PCA_comp):
                sum_energy+=self.beta_PCA*self.J[trial_aa,self.sequence[self.L+i],site,self.L+i]
        for j in range(self.L):
            if j == site:
                continue
            aa_j = self.sequence[j]
            sum_energy += self.beta * self.J[trial_aa, aa_j, site, j] # check indexing
            #sum_energy += self.J[aa_j, trial_aa, j, site] 
        prob = sum_energy  
        return prob
    
    def plm_site_distribution(self, site):
        """
        Compute probability distriution for specific site (normalized)
        """
        probs = []
        for trial_aa in range(21):
            probs.append(self.plm_calc(site, trial_aa))
        probs = np.array(probs)
        probs = np.exp(probs-probs.max()) #to avoid overflow for high beta
        probs /= probs.sum()
        return probs
    
    def draw_aa(self, site):
        """
        Sample a new AA at the given site from PLM distribution
        """
        probs = self.plm_site_distribution(site)
        new_aa = np.random.choice(21, p=probs) # aa from 0 to 20
        self.sequence[site] = new_aa

    def update_PCA_coords(self, model, plot=False):
        """
        Jointly update both PCA coordinates using Boltzmann sampling
        based on PLM-derived J_PCA tensor and the current amino acid sequence.

        Assumes: nb_PCA_comp == 2
        """
        #if self.J_PCA is None:
        #    raise ValueError("J_PCA not provided in model.")
        #if self.nb_PCA_comp != 2:
        #    raise ValueError(f"Joint 2D PCA update only supports 2 PCA components, got {self.nb_PCA_comp}.")

        L = self.L
        Nbins = 35
        offset = L  # Start index of PCA coords in self.sequence
        probs_2D = np.zeros((Nbins, Nbins))
        energies_2D = self.compute_coord_energy(model)

        # Numerically stable softmax
        shifted_energies = self.beta_PCA * energies_2D #(energy should be -)
        #shifted_energies -= shifted_energies.max()

        shifted_energies -= shifted_energies.max()  # subtract max for numerical stability

        probs_2D = np.exp(-shifted_energies)
        total = probs_2D.sum()

        # Check for numerical issues
        if total == 0 or np.isnan(total) or np.isinf(total):
            probs_2D = np.ones_like(probs_2D) / probs_2D.size  # uniform distribution
        else:
            probs_2D /= total

        # Sample from joint distribution
        flat_probs = probs_2D.flatten()
        choice = np.random.choice(Nbins * Nbins, p=flat_probs)
        i_sampled, j_sampled = np.unravel_index(choice, (Nbins, Nbins))

        self.sequence[offset + 0] = i_sampled  # PCA comp 0
        self.sequence[offset + 1] = j_sampled  # PCA comp 1

        return np.array([i_sampled, j_sampled])
    
    def compute_coord_energy(self, model):
        """
        model = 1,2,3
        """
        L = self.L
        Nbins = 35
        offset = L 
        energies_2D = np.zeros((Nbins, Nbins))
        energies_flat = np.zeros((Nbins*Nbins))
        if model == 1:
            # for i in range(Nbins):  # PCA comp 0
            #     for j in range(Nbins):  # PCA comp 1
            #         energy = 0.0
            #         for pos in range(L):  # real amino acid sites
            #             aa = self.sequence[pos]  # actual residue index (0–20)
            #             # Interact with PCA comp 0 (stored at position L)
            #             pca_aa0 = i  # interpreted as "amino acid index" at pos = L
            #             energy += self.J_PCA[aa, pca_aa0, pos, 0]

            #             # Interact with PCA comp 1 (stored at position L+1)
            #             pca_aa1 = j
            #             energy += self.J_PCA[aa, pca_aa1, pos, 1]
            # # Scale by beta_PCA (if used only for PCA couplings)
            #         energies_2D[i, j] = self.beta_PCA * energy
            ##Comment j'aurais fait
            for i in range(Nbins):  # PCA comp 0
                for j in range(Nbins):  # PCA comp 1
                    energy = 0.0
                    self.sequence[-2]=i
                    self.sequence[-1]=j
                    for pos in range(L+self.nb_PCA_comp):  # real amino acid sites
                        aa = self.sequence[pos]  # actual residue index (0–20)
                        # Interact with PCA comp 0 (stored at position L)
                        pca_aa0 = i  # interpreted as "amino acid index" at pos = L
                        if pos!=L :
                            energy += self.J[aa, pca_aa0, pos, -2]

                        # Interact with PCA comp 1 (stored at position L+1)
                        pca_aa1 = j
                        if pos != L+1:
                            energy += self.J[aa, pca_aa1, pos, -1]

            # Scale by beta_PCA (if used only for PCA couplings)
                    energies_2D[i, j] = self.beta_PCA * energy

        elif model == 2:
        # Compute energy for each (i,j) PCA coordinate pair
            for i in range(Nbins):  # PCA component 0
                for j in range(Nbins):  # PCA component 1
                    energy = 0.0
                    for pos in range(L):
                        aa = self.sequence[pos]
                        energy += self.beta_PCA * (self.J_PCA[aa, i, pos, 0] + self.J_PCA[aa, j, pos, 1])
                    energies_2D[i, j] = energy  # Store energy for visualization

        elif model == 3:
            for i in range(Nbins):
                energy = 0.0
                for pos in range(L):
                    aa = self.sequence[pos]
                    energy += self.beta_PCA * (self.J_PCA[aa, i, pos])
                energies_flat[i] = energy.item()
            energies_2D = energies_flat.reshape((Nbins, Nbins))
        energies_2D = -energies_2D  # Negate to match Boltzmann distribution (lower energy = higher probability)
        return energies_2D

    def seq_energy(self):
        sum=0
        for i in range(self.L):
            for j in range(self.L):
                sum+=self.J[self.sequence[i], self.sequence[j],i,j]
        return sum
    



class BatchSequencePLM:
    def __init__(self, J, N, beta=1, nb_PCA_comp=0, PCA_component_list=None, J_tens_PCA=None, beta_PCA=1):
        """
        Initialize with a batch of N independent sequences.
        """
        self.J = J
        self.J_PCA = J_tens_PCA
        self.beta = beta
        self.beta_PCA = beta_PCA
        self.nb_PCA_comp = nb_PCA_comp
        self.N = N

        if self.J_PCA is not None:
            self.L = J.shape[-1] - nb_PCA_comp
        else:
            self.L = J.shape[-1] - nb_PCA_comp

        # Initialize N random sequences
        core_sequences = np.random.randint(0, 21, size=(N, self.L))

        if nb_PCA_comp > 0:
            if PCA_component_list is None:
                raise ValueError("PCA_component_list must be provided for PCA components.")
            if len(PCA_component_list) != nb_PCA_comp:
                raise ValueError("Mismatch between number of PCA components and list provided.")
            pca_array = np.tile(PCA_component_list, (N, 1))  # replicate for each sequence
            self.sequences = np.concatenate((core_sequences, pca_array), axis=1)
        else:
            self.sequences = core_sequences  # shape: (N, L [+ nb_PCA])

    def plm_calc_batch(self, site, trial_aa):
        """
        Compute unnormalized pseudo-likelihood for all N sequences at one site and one trial AA.
        Return: (N,) array
        """
        aa_j_all = self.sequences[:, :self.L]  # shape (N, L)
        aa_trial = np.full(self.N, trial_aa)

        energy = np.zeros(self.N)

        for j in range(self.L):
            if j == site:
                continue
            aa_j = aa_j_all[:, j]  # shape (N,)
            energy += self.beta * np.asarray(self.J[trial_aa, aa_j, site, j])  # vectorized lookup

        # If PCA is used
        if self.nb_PCA_comp > 0:
            for i in range(self.nb_PCA_comp):
                PCA_coord = self.sequences[:, self.L + i]
                if self.J_PCA is not None:
                    energy += self.beta_PCA * np.asarray(self.J_PCA[trial_aa, PCA_coord, site, i])
                else:
                    energy += self.beta_PCA * np.asarray(self.J[trial_aa, PCA_coord, site, self.L + i])
        
        return energy

    def plm_site_distribution_batch(self, site):
        """
        Compute PLM probabilities for each AA (0..20) for all N sequences at site.
        Returns: (N, 21) array of probabilities
        """
        raw_scores = np.zeros((self.N, 21))
        for aa in range(21):
            raw_scores[:, aa] = self.plm_calc_batch(site, aa)

        raw_scores -= raw_scores.max(axis=1, keepdims=True)  # stability
        probs = np.exp(raw_scores)
        probs /= probs.sum(axis=1, keepdims=True)  # normalize
        return probs  # shape: (N, 21)

    def draw_aa_batch(self, site):
        """
        Sample new AAs for all N sequences at a given site.
        """
        probs = self.plm_site_distribution_batch(site)  # (N, 21)
        new_aas = np.array([np.random.choice(21, p=p) for p in probs])  # (N,)
        self.sequences[:, site] = new_aas

    def evolve_all(self, n_iter=500):
        """
        Perform Gibbs sampling over all sequences for n_iter iterations.
        """
        for _ in tqdm(range(n_iter)):
            site=np.random.choice(self.L)
            self.draw_aa_batch(site)

    def get_sequences(self):
        """
        Return all N sequences.
        """
        return self.sequences

    def to_letters(self):
        """
        Return N sequences in letter format.
        """
        num_to_letter = {v: k for k, v in letter_to_num.items()}
        letter_seqs = []
        for seq in self.sequences[:, :self.L]:
            letter_seq = ''.join([num_to_letter[i] for i in seq])
            letter_seqs.append(letter_seq)
        return letter_seqs

In [24]:
generated_seqs = generate_plm(Jtens.cpu().numpy(), N_seqs=100000, beta=1, nb_PCA_comp=2, beta_pca=1, target_coords=np.array([24+21, 20+21]))

  0%|          | 0/100000 [00:00<?, ?it/s]


IndexError: index 66 is out of bounds for axis 1 with size 56

## Plots

In [21]:
from plm_gen_methods import generate_plm_n_save, generate_coords_n_save, generate_plm_vect_n_save
from seq_utils import read_tensor_from_txt, set_seed, letters_to_nums, modify_seq, find_target_seq, sequences_from_fasta
from PCA_func import plot_projected_pca, plot_pca_of_sequences, plot_two_pca_side_by_side, plot_projected_pca_mult, plot_projected_pca_time

### Load train sequences

In [23]:
older_dir = os.path.abspath(os.path.join(current_dir, '..', '..', '..'))
Z,W = quickread(older_dir + '/DataAttentionDCA/jdoms/jdoms_bacteria_train2.fasta')

Total sequences read: 14502
Sequences after filtering: 14502
Sampling 100000 pairs out of 105146751 total pairs.
Mean fraction of identical positions (sampled): 0.37239746031746035
Computed theta: 0.3265328391239263


100%|██████████| 14502/14502 [00:08<00:00, 1644.46it/s]

3265.70225762244


In [ ]:
plot_projected_pca_time(Z.T,generated_seqs, target_coords=np.array([24,20]), Nbins=35)